In [ ]:
# ==============================================================================
# STEP 0: DEPENDENCY INSTALLATION
# ==============================================================================
!pip install --upgrade -q lightgbm scikit-learn pandas numpy
print("Dependencies verified successfully!")

In [ ]:
# ==============================================================================
# AMAZON ML CHALLENGE 2026 - BUSINESS ENTITY RESOLUTION PIPELINE
# STREAMING LOW-RAM VERSION (Auto-detects Colab/Kaggle, <3 GB RAM Footprint)
# ==============================================================================

import os
import re
import gc
import sys
import pickle
import warnings
import unicodedata
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb

# Suppress benign Scikit-Learn / LightGBM feature-name warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# ------------------------------------------------------------------------------
# SECTION 1: PATH DETECTION (Auto-detects Colab, Kaggle, or Local)
# ------------------------------------------------------------------------------
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset"
KAGGLE_ALT_INPUT = "/kaggle/input/amazon-ml-challenge-2026/dataset"
COLAB_DIR = "/content/dataset"

if os.path.exists(os.path.join(COLAB_DIR, "train")):
    BASE_DIR = COLAB_DIR
    OUTPUT_DIR = "/content/output"
    CHECKPOINT_DIR = "/content/checkpoints"
elif os.path.exists(os.path.join(KAGGLE_INPUT_DIR, "train")):
    BASE_DIR = KAGGLE_INPUT_DIR
    OUTPUT_DIR = "/kaggle/working/outputs"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
elif os.path.exists(os.path.join(KAGGLE_ALT_INPUT, "train")):
    BASE_DIR = KAGGLE_ALT_INPUT
    OUTPUT_DIR = "/kaggle/working/outputs"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
else:
    BASE_DIR = "student_resource/dataset"
    OUTPUT_DIR = "output"
    CHECKPOINT_DIR = "checkpoints"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"[Paths] Train Dir:      {TRAIN_DIR}")
print(f"[Paths] Test Dir:       {TEST_DIR}")
print(f"[Paths] Output Dir:     {OUTPUT_DIR}")
print(f"[Paths] Checkpoint Dir: {CHECKPOINT_DIR}")

# ------------------------------------------------------------------------------
# SECTION 2: TEXT NORMALIZATION & SIMILARITY EXTRACTION
# ------------------------------------------------------------------------------
LEGAL_SUFFIXES = {
    r'\b(private limited|pvt ltd|pvt\.? ltd\.?|limited|ltd\.?)\b': ' pvt ltd ',
    r'\b(corporation|corp\.?|incorporated|inc\.?)\b': ' inc ',
    r'\b(limited liability company|llc|l\.l\.c\.)\b': ' llc ',
    r'\b(limited liability partnership|llp|l\.l\.p\.)\b': ' llp ',
    r'\b(societe anonyme|s\.a\.|sa)\b': ' sa ',
    r'\b(societe a responsabilite limitee|sarl|s\.a\.r\.l\.)\b': ' sarl ',
    r'\b(societe par actions simplifiee|sas|s\.a\.s\.)\b': ' sas ',
}

ADDRESS_EXPANSIONS = {
    r'\b(st|st\.)\b': 'street',
    r'\b(rd|rd\.)\b': 'road',
    r'\b(ave|ave\.)\b': 'avenue',
    r'\b(blvd|blvd\.)\b': 'boulevard',
    r'\b(dr|dr\.)\b': 'drive',
    r'\b(ln|ln\.)\b': 'lane',
    r'\b(hwy|hwy\.)\b': 'highway',
    r'\b(apt|apt\.|apartment)\b': 'unit',
    r'\b(fl|fl\.|floor)\b': 'floor',
    r'\b(rue|r\.)\b': 'rue',
    r'\b(av|av\.)\b': 'avenue',
    r'\b(bld|bd)\b': 'boulevard',
}

FEATURE_NAMES = [
    'tfidf_sim', 'name_jac', 'addr_jac', 'name_ngram', 'addr_ngram',
    'overlap_tokens', 'len_ratio_name', 'exact_name', 'exact_addr',
    'addr_missing', 'is_substr'
]

def normalize_text(text, is_address=False):
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    for pattern, repl in LEGAL_SUFFIXES.items():
        text = re.sub(pattern, repl, text)
    if is_address:
        for pattern, repl in ADDRESS_EXPANSIONS.items():
            text = re.sub(pattern, repl, text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def token_jaccard(str1, str2):
    s1, s2 = set(str1.split()), set(str2.split())
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

def char_ngram_jaccard(str1, str2, n=3):
    if len(str1) < n or len(str2) < n:
        return 0.0
    s1 = {str1[i:i+n] for i in range(len(str1)-n+1)}
    s2 = {str2[i:i+n] for i in range(len(str2)-n+1)}
    return len(s1 & s2) / len(s1 | s2) if (s1 | s2) else 0.0

def extract_pair_features(s1_name, s1_addr, tgt_name, tgt_addr, tfidf_sim):
    name_jac = token_jaccard(s1_name, tgt_name)
    addr_jac = token_jaccard(s1_addr, tgt_addr) if (s1_addr and tgt_addr) else 0.0
    name_ngram = char_ngram_jaccard(s1_name, tgt_name, n=3)
    addr_ngram = char_ngram_jaccard(s1_addr, tgt_addr, n=3) if (s1_addr and tgt_addr) else 0.0
    s1_tokens = set(s1_name.split())
    tgt_tokens = set(tgt_name.split())
    overlap_tokens = len(s1_tokens & tgt_tokens)
    len_ratio_name = min(len(s1_name), len(tgt_name)) / max(len(s1_name), len(tgt_name), 1)
    exact_name = 1.0 if s1_name == tgt_name and s1_name != "" else 0.0
    exact_addr = 1.0 if s1_addr == tgt_addr and s1_addr != "" else 0.0
    addr_missing = 1.0 if (s1_addr == "" or tgt_addr == "") else 0.0
    is_substr = 1.0 if (s1_name in tgt_name or tgt_name in s1_name) and min(len(s1_name), len(tgt_name)) > 4 else 0.0

    return [
        tfidf_sim, name_jac, addr_jac, name_ngram, addr_ngram,
        overlap_tokens, len_ratio_name, exact_name, exact_addr,
        addr_missing, is_substr
    ]

# ------------------------------------------------------------------------------
# SECTION 3: MODEL TRAINING (With Checkpointing)
# ------------------------------------------------------------------------------
MODEL_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "lightgbm_model.pkl")

if os.path.exists(MODEL_CHECKPOINT):
    print(f"Found existing model checkpoint at {MODEL_CHECKPOINT}. Loading model...")
    with open(MODEL_CHECKPOINT, 'rb') as f:
        clf = pickle.load(f)
else:
    print("\n--- Step 1: Training LightGBM Matcher ---")
    SAMPLE_TRAIN_S1 = 50000
    df_s1_tr = pd.read_csv(os.path.join(TRAIN_DIR, "train_source1.tsv"), sep="\t", nrows=SAMPLE_TRAIN_S1)
    df_s2_tr = pd.read_csv(os.path.join(TRAIN_DIR, "train_source2.tsv"), sep="\t", nrows=SAMPLE_TRAIN_S1*2)
    df_s3_tr = pd.read_csv(os.path.join(TRAIN_DIR, "train_source3.tsv"), sep="\t", nrows=SAMPLE_TRAIN_S1*2)
    df_gt = pd.read_csv(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"), sep="\t")

    gt_map = {}
    for _, row in df_gt.iterrows():
        s1_id = row['source1_entity_id']
        matches = set(str(row['matched_entity_ids']).split(',')) if pd.notna(row['matched_entity_ids']) and str(row['matched_entity_ids']).strip() else set()
        gt_map[s1_id] = matches

    for df in [df_s1_tr, df_s2_tr, df_s3_tr]:
        df['clean_name'] = df['business_name'].apply(normalize_text)
        df['clean_address'] = df['business_address'].apply(lambda x: normalize_text(x, is_address=True))

    df_tgt_tr = pd.concat([df_s2_tr, df_s3_tr], ignore_index=True)
    s1_names_tr = df_s1_tr['clean_name'].tolist()
    s1_addrs_tr = df_s1_tr['clean_address'].tolist()
    s1_ids_tr = df_s1_tr['entity_id'].tolist()
    
    tgt_names_tr = df_tgt_tr['clean_name'].tolist()
    tgt_addrs_tr = df_tgt_tr['clean_address'].tolist()
    tgt_ids_tr = df_tgt_tr['entity_id'].tolist()

    X_train, y_train = [], []
    for country in df_s1_tr['country'].unique():
        s1_mask = (df_s1_tr['country'] == country).values
        tgt_mask = (df_tgt_tr['country'] == country).values
        
        if not np.any(s1_mask) or not np.any(tgt_mask):
            continue
            
        s1_sub_idx = np.where(s1_mask)[0]
        tgt_sub_idx = np.where(tgt_mask)[0]
        
        tgt_texts = [tgt_names_tr[i] + " " + tgt_addrs_tr[i] for i in tgt_sub_idx]
        s1_texts = [s1_names_tr[i] + " " + s1_addrs_tr[i] for i in s1_sub_idx]
        
        vec = TfidfVectorizer(ngram_range=(1, 2), analyzer='word', max_features=70000, sublinear_tf=True)
        tgt_mat = vec.fit_transform(tgt_texts)
        s1_mat = vec.transform(s1_texts)
        
        batch_size = 500
        for i in range(0, len(s1_sub_idx), batch_size):
            sub_batch_s1 = s1_mat[i:i+batch_size]
            batch_sims = sub_batch_s1.dot(tgt_mat.T).toarray()
            
            for r_idx in range(len(batch_sims)):
                orig_s1_idx = s1_sub_idx[i + r_idx]
                cur_s1 = s1_ids_tr[orig_s1_idx]
                sim_row = batch_sims[r_idx]
                top_k = min(30, len(tgt_sub_idx))
                top_indices = np.argpartition(sim_row, -top_k)[-top_k:]
                true_set = gt_map.get(cur_s1, set())
                
                s1_n = s1_names_tr[orig_s1_idx]
                s1_a = s1_addrs_tr[orig_s1_idx]
                
                for idx in top_indices:
                    sim_val = float(sim_row[idx])
                    if sim_val >= 0.05:
                        orig_tgt_idx = tgt_sub_idx[idx]
                        tid = tgt_ids_tr[orig_tgt_idx]
                        feats = extract_pair_features(
                            s1_n, s1_a,
                            tgt_names_tr[orig_tgt_idx], tgt_addrs_tr[orig_tgt_idx],
                            sim_val
                        )
                        X_train.append(feats)
                        y_train.append(1 if tid in true_set else 0)
            del batch_sims

    X_train = pd.DataFrame(X_train, columns=FEATURE_NAMES)
    y_train = np.array(y_train)
    print(f"Training set built: {X_train.shape[0]} pairs (Positive: {np.mean(y_train):.4f})")

    clf = lgb.LGBMClassifier(
        n_estimators=150,
        learning_rate=0.07,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    clf.fit(X_train, y_train)
    
    with open(MODEL_CHECKPOINT, 'wb') as f:
        pickle.dump(clf, f)
    print(f"Saved trained model to {MODEL_CHECKPOINT}")

    del df_s1_tr, df_s2_tr, df_s3_tr, df_tgt_tr, X_train, y_train
    gc.collect()

# ------------------------------------------------------------------------------
# SECTION 4: STREAMING PER-COUNTRY TEST PROCESSING
# ------------------------------------------------------------------------------
print("\n--- Step 2: Processing Test Partitions ---")
DECISION_TAU = 0.72

# Test Country order
COUNTRIES = ['France', 'US', 'India']

for country in COUNTRIES:
    country_tag = country.replace(' ', '_')
    country_ckpt = os.path.join(CHECKPOINT_DIR, f"preds_{country_tag}.pkl")
    
    if os.path.exists(country_ckpt):
        print(f"Country '{country}' already completed. Using checkpoint {country_ckpt}")
        continue
        
    print(f"\nLoading test data for Country: '{country}'...")
    
    # Load and filter only the current country into memory
    s1_rows, tgt_rows = [], []
    
    # Stream test_source1 for country
    for chunk in pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="\t", chunksize=100000):
        c_chunk = chunk[chunk['country'] == country]
        if len(c_chunk) > 0:
            s1_rows.append(c_chunk)
    df_s1_c = pd.concat(s1_rows, ignore_index=True) if s1_rows else pd.DataFrame()
    del s1_rows
    gc.collect()
    
    # Stream test_source2 & test_source3 for country
    for fname in ["test_source2.tsv", "test_source3.tsv"]:
        for chunk in pd.read_csv(os.path.join(TEST_DIR, fname), sep="\t", chunksize=100000):
            c_chunk = chunk[chunk['country'] == country]
            if len(c_chunk) > 0:
                tgt_rows.append(c_chunk)
    df_tgt_c = pd.concat(tgt_rows, ignore_index=True) if tgt_rows else pd.DataFrame()
    del tgt_rows
    gc.collect()
    
    print(f"[{country}] S1 count: {len(df_s1_c)} | Target pool: {len(df_tgt_c)}")
    
    c_matching = defaultdict(list)
    c_candidates = defaultdict(list)
    
    if len(df_s1_c) > 0 and len(df_tgt_c) > 0:
        s1_names = df_s1_c['business_name'].apply(normalize_text).tolist()
        s1_addrs = df_s1_c['business_address'].apply(lambda x: normalize_text(x, is_address=True)).tolist()
        s1_ids = df_s1_c['entity_id'].tolist()
        del df_s1_c
        gc.collect()
        
        tgt_names = df_tgt_c['business_name'].apply(normalize_text).tolist()
        tgt_addrs = df_tgt_c['business_address'].apply(lambda x: normalize_text(x, is_address=True)).tolist()
        tgt_ids = df_tgt_c['entity_id'].tolist()
        del df_tgt_c
        gc.collect()
        
        tgt_texts = [tgt_names[i] + " " + tgt_addrs[i] for i in range(len(tgt_ids))]
        s1_texts = [s1_names[i] + " " + s1_addrs[i] for i in range(len(s1_ids))]
        
        vec = TfidfVectorizer(ngram_range=(1, 2), analyzer='word', max_features=70000, sublinear_tf=True)
        tgt_mat = vec.fit_transform(tgt_texts)
        del tgt_texts
        gc.collect()
        
        s1_mat = vec.transform(s1_texts)
        del s1_texts, vec
        gc.collect()
        
        batch_size = 500
        for i in range(0, len(s1_ids), batch_size):
            if i % 25000 == 0:
                print(f"  [{country}] Progress: {i}/{len(s1_ids)} records...")
                
            sub_batch_s1 = s1_mat[i:i+batch_size]
            batch_sims = sub_batch_s1.dot(tgt_mat.T).toarray()
            
            pair_feats_batch = []
            pair_records = []
            
            for r_idx in range(len(batch_sims)):
                s1_idx = i + r_idx
                cur_s1 = s1_ids[s1_idx]
                s1_n = s1_names[s1_idx]
                s1_a = s1_addrs[s1_idx]
                sim_row = batch_sims[r_idx]
                top_k = min(30, len(tgt_ids))
                top_indices = np.argpartition(sim_row, -top_k)[-top_k:]
                top_indices = top_indices[np.argsort(-sim_row[top_indices])]
                
                for idx in top_indices:
                    sim_val = float(sim_row[idx])
                    if sim_val >= 0.05:
                        tid = tgt_ids[idx]
                        c_candidates[cur_s1].append(tid)
                        feats = extract_pair_features(
                            s1_n, s1_a,
                            tgt_names[idx], tgt_addrs[idx],
                            sim_val
                        )
                        pair_feats_batch.append(feats)
                        pair_records.append((cur_s1, tid))
                        
            if len(pair_feats_batch) > 0:
                df_batch_feats = pd.DataFrame(pair_feats_batch, columns=FEATURE_NAMES)
                probs = clf.predict_proba(df_batch_feats)[:, 1]
                for (s1_elem, tid), prob in zip(pair_records, probs):
                    if prob >= DECISION_TAU:
                        c_matching[s1_elem].append(tid)
                        
            del batch_sims
            
        del tgt_mat, s1_mat, s1_names, s1_addrs, s1_ids, tgt_names, tgt_addrs, tgt_ids
        gc.collect()
        
    with open(country_ckpt, 'wb') as f:
        pickle.dump((dict(c_matching), dict(c_candidates)), f)
    print(f"Saved results for '{country}' to {country_ckpt}")

# ------------------------------------------------------------------------------
# SECTION 5: MERGE ALL CHECKPOINTS & EXPORT TSV FILES
# ------------------------------------------------------------------------------
print("\n--- Step 3: Aggregating Checkpoints & Writing Final Output TSVs ---")
matching_dict = {}
candidate_dict = {}

for country in COUNTRIES:
    country_tag = country.replace(' ', '_')
    country_ckpt = os.path.join(CHECKPOINT_DIR, f"preds_{country_tag}.pkl")
    if os.path.exists(country_ckpt):
        with open(country_ckpt, 'rb') as f:
            c_match, c_cand = pickle.load(f)
            matching_dict.update(c_match)
            candidate_dict.update(c_cand)

df_s1_full = pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="\t", usecols=['entity_id'])
all_s1_ids = df_s1_full['entity_id'].tolist()
del df_s1_full
gc.collect()

matching_tsv_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
candidate_tsv_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

with open(matching_tsv_path, 'w', encoding='utf-8') as f_match, \
     open(candidate_tsv_path, 'w', encoding='utf-8') as f_cand:
    
    f_match.write("source1_entity_id\tmatched_entity_ids\n")
    f_cand.write("source1_entity_id\tcandidate_entity_ids\n")
    
    for s1_id in all_s1_ids:
        cands = list(dict.fromkeys(candidate_dict.get(s1_id, [])))
        matches = list(dict.fromkeys(matching_dict.get(s1_id, [])))
        
        cand_set = set(cands)
        matches = [m for m in matches if m in cand_set]
        
        f_match.write(f"{s1_id}\t{','.join(matches)}\n")
        f_cand.write(f"{s1_id}\t{','.join(cands)}\n")

print(f"\nSUCCESS! Written {len(all_s1_ids)} rows to:")
print(f"  1. {matching_tsv_path}")
print(f"  2. {candidate_tsv_path}")
